# Phase 3: Advanced Models & Imbalance Handling

**Author:** [Your Name]  
**Last run:** [Date]  
**Goal:** Compare advanced models and address class imbalance.  
**Input:** `../../data/processed/stroke_cleaned.csv`  
**Output:** `../../results/metrics/phase3_model_comparison.csv`

> **Data leakage reminder:** SMOTE and BMI imputation must be fit *only* on training data.
> The pipeline below handles this automatically via `sklearn.pipeline.Pipeline`.

### Checklist
- [ ] Imbalance strategy comparison (class weights vs SMOTE)
- [ ] Random Forest
- [ ] Gradient Boosting (XGBoost or LightGBM)
- [ ] SVM or KNN
- [ ] Cross-validation with StratifiedKFold
- [ ] Feature importance plots
- [ ] Full model comparison table

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (classification_report, confusion_matrix,
                              roc_auc_score, RocCurveDisplay)
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

sns.set_theme(style='whitegrid')
RANDOM_STATE = 42

PROCESSED_PATH = os.path.join('..', '..', 'data', 'processed', 'stroke_cleaned.csv')
METRICS_PATH   = os.path.join('..', '..', 'results', 'metrics', 'phase3_model_comparison.csv')
FIGURES_PATH   = os.path.join('..', '..', 'results', 'figures')

## 1. Load Data & Split

In [ ]:
df = pd.read_csv(PROCESSED_PATH)
X = df.drop(columns=['stroke'])
y = df['stroke']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)
print(f'Train: {X_train.shape}  |  Test: {X_test.shape}')
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

## 2. Evaluation Helper

In [ ]:
results = []

def evaluate_pipeline(name, pipeline):
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    y_prob = pipeline.predict_proba(X_test)[:, 1]
    auc = roc_auc_score(y_test, y_prob)
    report = classification_report(y_test, y_pred, output_dict=True)
    print(f'\n--- {name} ---')
    print(classification_report(y_test, y_pred))
    print(f'ROC-AUC: {auc:.4f}')
    results.append({
        'model': name,
        'precision_stroke': report['1']['precision'],
        'recall_stroke':    report['1']['recall'],
        'f1_stroke':        report['1']['f1-score'],
        'roc_auc':          auc,
        'accuracy':         report['accuracy'],
    })
    return pipeline

## 3. Random Forest — class_weight='balanced'

In [ ]:
rf_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', RandomForestClassifier(n_estimators=200, class_weight='balanced',
                                    random_state=RANDOM_STATE))
])
rf_fitted = evaluate_pipeline('Random Forest (balanced)', rf_pipe)

## 4. Random Forest — SMOTE oversampling

In [ ]:
rf_smote_pipe = ImbPipeline([
    ('scaler', StandardScaler()),
    ('smote', SMOTE(random_state=RANDOM_STATE)),
    ('clf', RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE))
])
evaluate_pipeline('Random Forest + SMOTE', rf_smote_pipe)

## 5. Gradient Boosting

In [ ]:
gb_pipe = ImbPipeline([
    ('scaler', StandardScaler()),
    ('smote', SMOTE(random_state=RANDOM_STATE)),
    ('clf', GradientBoostingClassifier(n_estimators=200, learning_rate=0.05,
                                        random_state=RANDOM_STATE))
])
evaluate_pipeline('Gradient Boosting + SMOTE', gb_pipe)

## 6. SVM

In [ ]:
svm_pipe = ImbPipeline([
    ('scaler', StandardScaler()),
    ('smote', SMOTE(random_state=RANDOM_STATE)),
    ('clf', SVC(class_weight='balanced', probability=True, random_state=RANDOM_STATE))
])
evaluate_pipeline('SVM + SMOTE', svm_pipe)

## 7. Feature Importance (Random Forest)

In [ ]:
importances = rf_fitted.named_steps['clf'].feature_importances_
feat_df = pd.Series(importances, index=X.columns).sort_values(ascending=False).head(15)

fig, ax = plt.subplots(figsize=(8, 5))
feat_df.plot(kind='bar', ax=ax, color='steelblue')
ax.set_title('Top 15 Feature Importances — Random Forest')
ax.set_ylabel('Importance')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_PATH, 'p3_feature_importance_rf.png'), dpi=150)
plt.show()

## 8. Model Comparison Table

In [ ]:
metrics_df = pd.DataFrame(results).sort_values('recall_stroke', ascending=False)
os.makedirs(os.path.dirname(METRICS_PATH), exist_ok=True)
metrics_df.to_csv(METRICS_PATH, index=False)
display(metrics_df)